# 測定方法の改善パイロット
質問形式・学習例拡張・X/Y距離を比較。3ペア、108行動入力、新評価文36件。PLAN.mdに実行前の設計を固定。

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install transformers==4.57.6 scikit-learn scipy pandas numpy


In [ ]:
from pathlib import Path
import subprocess,sys,json
REPO=Path('/content/GraduationThesis-revision')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ryouy/GraduationThesis.git',str(REPO)],check=True)
OUT2=Path('/content/revision-pilot-20261103')
OUT2.mkdir(exist_ok=True)
p=subprocess.Popen([sys.executable,'-u','1103_revision/run_revision.py','--out',str(OUT2)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
with (OUT2/'run.log').open('w') as log:
    for line in p.stdout:
        log.write(line);log.flush();print(line,end='',flush=True)
assert p.wait()==0


In [ ]:
import pandas as pd
u=pd.read_csv(OUT2/'probe_auc.csv')
a=pd.read_csv(OUT2/'answers.csv')
print(u.query('style=="all" and layer<24').groupby(['concept','training']).auc.mean().unstack())
print(a[a.level=='near'].groupby('mode').target_mention.agg(['sum','count']))
subprocess.run([sys.executable,'1103_revision/verify_results.py',str(OUT2)],cwd=REPO,check=True)


In [ ]:
print('REVISION_EXPORT_START')
print(json.dumps({p.name:p.read_text() for p in sorted(OUT2.iterdir()) if p.is_file()}))
print('REVISION_EXPORT_END')